# Human-in-the-loop

`HumanInTheLoopMiddleware` pauses selected tool calls before execution. A checkpointer and thread ID retain the pending action until a reviewer answers. `approve` executes the original call, `edit` executes changed arguments, `reject` returns an error without execution, and `respond` supplies a successful tool result without execution.

A batched interrupt contains an ordered list of actions and takes decisions in that same order. Per-call mode creates a separate interrupt ID and response schema for each action, allowing independent review. A `when` predicate can restrict approval to calls matching an argument-level condition.

[Human-in-the-loop](https://docs.langchain.com/oss/python/langchain/human-in-the-loop)

In [ ]:
# Imports and project path setup only; examples start in the next cell.
from pathlib import Path
import sys

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
LESSONS = ROOT / "notebooks" / "langchain" / "06_middlewares"
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
if str(LESSONS) not in sys.path:
    sys.path.insert(0, str(LESSONS))

from lab.data import prompt
from lab.display import TraceMiddleware, show_messages
from lab.tools import draft_count, save_draft
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command
from model_config import get_model, run_config
from langchain.agents.middleware import ToolCallRequest
from langchain.tools import tool
from langchain_core.messages import AIMessage
from lab.faults import ScriptedModel, tool_request
from pydantic import ValidationError
from lab.tools import lookup_order
from lab.display import expect_error


### Approve, edit or reject a draft

Safe order lookup runs without review. Draft creation pauses. A custom description identifies the recipient and subject; `edit_notice` tells the model which arguments were replaced so it does not repeat the original action.

The resumed run streams completed node updates. The same checkpoint supplies the final conversation after the stream finishes; streaming changes delivery, not the approval policy.

In [ ]:
def describe_draft(tool_call, state, runtime):
    args = tool_call["args"]
    return f"Review draft to {args['to']} with subject {args['subject']}"

approval_agent = create_agent(
    model=get_model(), tools=[save_draft, lookup_order],
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "save_draft": {"allowed_decisions": ["approve", "edit", "reject"], "description": describe_draft},
                "lookup_order": False,
            },
            description_prefix="Review requested",
            edit_notice="The reviewer changed this draft. Do not repeat the original action.",
        ),
        TraceMiddleware("05_approval", pretty=True),
    ],
    checkpointer=InMemorySaver(),
)
approval_config = run_config("human-review")
before_approval = draft_count()
approval_paused = approval_agent.invoke(
    {"messages": [("user", "Look up order DEMO-42, then use save_draft to save one draft to ala.nowak@example.com with subject Workshop and body See you on Friday.")]},
    config=approval_config,
)
assert approval_paused.get("__interrupt__"), "The model did not request the draft tool."
assert draft_count() == before_approval

In [ ]:
# Inspect output
show_messages(approval_paused["messages"], pretty=True)
print(approval_paused["__interrupt__"][0].value)

In [ ]:
# Execute
choice = input("Draft decision (approve / edit / reject): ").strip().lower()
if choice not in {"approve", "edit", "reject"}:
    raise ValueError("Choose approve, edit or reject; the draft remains paused.")
decision = {"type": choice}
if choice == "edit":
    decision["edited_action"] = {"name": "save_draft", "args": {
        "to": input("To: "), "subject": input("Subject: "), "body": input("Body: "),
    }}
elif choice == "reject":
    decision["message"] = "The human rejected this draft. Do not create another draft."
approval_updates = list(approval_agent.stream(Command(resume={"decisions": [decision]}), config=approval_config, stream_mode="updates"))
approval_result = approval_agent.get_state(approval_config).values

In [ ]:
# Inspect output
show_messages(approval_result["messages"], pretty=True)
print("Drafts created:", draft_count() - before_approval)
print("Streamed nodes:", [name for update in approval_updates for name in update])

### Conditional interrupts

The predicate examines the tool arguments. Subjects containing `Artur` require review; other subjects execute directly. The predicate must return the same result on resume, including when a reviewer rejects the action.

In [ ]:
# Execute
def check_if_artur(request: ToolCallRequest) -> bool:
    return "artur" in request.tool_call["args"].get("subject", "").lower()

conditional_agent = create_agent(
    model=get_model(), tools=[save_draft],
    middleware=[HumanInTheLoopMiddleware(interrupt_on={
        "save_draft": {"allowed_decisions": ["approve", "reject"], "when": check_if_artur},
    }), TraceMiddleware("05_conditional", pretty=True)],
    checkpointer=InMemorySaver(),
)
conditional_config = run_config("conditional-review")
conditional_auto = conditional_agent.invoke({"messages": [("user", "Use save_draft exactly once: to ala.nowak@example.com, subject General workshop, body Friday at 10:00.")]}, config=conditional_config)
assert not conditional_auto.get("__interrupt__")

In [ ]:
# Inspect output
show_messages(conditional_auto["messages"], pretty=True)

In [ ]:
# Execute
before_conditional = draft_count()
conditional_paused = conditional_agent.invoke({"messages": [("user", "Use save_draft exactly once: to ala.nowak@example.com, subject Artur workshop, body Friday at 10:00.")]}, config=conditional_config)
assert conditional_paused.get("__interrupt__"), "The model did not request the gated draft."
assert draft_count() == before_conditional

In [ ]:
# Inspect output
show_messages(conditional_paused["messages"], pretty=True)
print(conditional_paused["__interrupt__"][0].value)

In [ ]:
# Execute
conditional_choice = input("Artur draft (approve / reject): ").strip().lower()
if conditional_choice not in {"approve", "reject"}:
    raise ValueError("Choose approve or reject; the draft remains paused.")
conditional_result = conditional_agent.invoke(Command(resume={"decisions": [{"type": conditional_choice, "message": "Do not repeat this draft if rejected."}]}), config=conditional_config)

In [ ]:
# Inspect output
show_messages(conditional_result["messages"], pretty=True)

### Respond as the tool

An `ask_user` call expects the human to supply information. A `respond` decision becomes its successful tool result, and the Python tool body does not execute. A refusal to authorize a side effect belongs in `reject`, not `respond`.

In [ ]:
# Execute
ask_user_executions = []

@tool
def ask_user(question: str) -> str:
    """Ask the human a question and wait for their answer."""
    ask_user_executions.append(question)
    return "No human response supplied."

question_agent = create_agent(
    model=get_model(), tools=[ask_user],
    middleware=[HumanInTheLoopMiddleware(interrupt_on={"ask_user": {"allowed_decisions": ["respond"]}})],
    checkpointer=InMemorySaver(),
)
question_config = run_config("human-answer")
question_paused = question_agent.invoke({"messages": [("user", "Use ask_user to ask me which day the workshop should take place. Do not choose a day yourself.")]}, config=question_config)
assert question_paused.get("__interrupt__")

In [ ]:
# Inspect output
show_messages(question_paused["messages"], pretty=True)
print(question_paused["__interrupt__"][0].value)

In [ ]:
# Execute
human_answer = input("Your answer: ")
question_result = question_agent.invoke(Command(resume={"decisions": [{"type": "respond", "message": human_answer}]}), config=question_config)
assert not ask_user_executions

In [ ]:
# Inspect output
show_messages(question_result["messages"], pretty=True)

### Two actions in one batched review

Two requested calls produce one interrupt. Every action needs one decision in the original action order. The scripted model requests the same two actions each time, so the difference between batched and per-call review is visible independently of model planning.

In [ ]:
# Execute
reviewed_labels = []

@tool
def record_review(label: str) -> str:
    """Record a reviewed label in this in-memory list."""
    reviewed_labels.append(label)
    return f"Recorded {label}"

def two_review_calls():
    return ScriptedModel(responses=[
        AIMessage(content="", tool_calls=[
            tool_request("record_review", label="first").tool_calls[0],
            tool_request("record_review", label="second").tool_calls[0],
        ]), AIMessage(content="Review complete."),
    ])

batch_agent = create_agent(model=two_review_calls(), tools=[record_review],
    middleware=[HumanInTheLoopMiddleware(interrupt_on={"record_review": {"allowed_decisions": ["approve", "reject"]}})], checkpointer=InMemorySaver())
batch_config = run_config("batched-review")
batch_paused = batch_agent.invoke({"messages": [("user", "Record both labels.")]}, config=batch_config)
batch_actions = batch_paused["__interrupt__"][0].value["action_requests"]
assert len(batch_actions) == 2

In [ ]:
# Inspect output
show_messages(batch_paused["messages"], pretty=True)
print("Ordered actions:", batch_actions)

In [ ]:
# Execute
batch_decisions = []
for action in batch_actions:
    batch_choice = input(f"{action['args']['label']} (approve / reject): ").strip().lower()
    if batch_choice not in {"approve", "reject"}:
        raise ValueError("Choose approve or reject; the batch remains paused.")
    batch_decisions.append({"type": batch_choice})
batch_result = batch_agent.invoke(Command(resume={"decisions": batch_decisions}), config=batch_config)

In [ ]:
# Inspect output
show_messages(batch_result["messages"], pretty=True)
print("Executed labels:", reviewed_labels)

### Per-call review, response schemas and partial resume

Each gated call has its own interrupt ID and allowed-response JSON schema. A malformed response fails validation before the tool executes. Answering one ID leaves the other action pending; the next resume supplies that action's own decision.

In [ ]:
# Execute
per_call_agent = create_agent(model=two_review_calls(), tools=[record_review],
    middleware=[HumanInTheLoopMiddleware(interrupt_on={"record_review": {"allowed_decisions": ["approve", "edit", "reject"]}}, interrupt_mode="per_call")], checkpointer=InMemorySaver())
per_call_config = run_config("per-call-review")
per_call_paused = per_call_agent.invoke({"messages": [("user", "Record both labels.")]}, config=per_call_config)
assert len(per_call_paused["__interrupt__"]) == 2
first_interrupt = per_call_paused["__interrupt__"][0]
with expect_error(ValidationError):
    per_call_agent.invoke(Command(resume={first_interrupt.id: {"type": "edit", "edited_action": {"name": "record_review", "args": {}}}}), config=per_call_config)

In [ ]:
# Inspect output
show_messages(per_call_paused["messages"], pretty=True)
for pending in per_call_paused["__interrupt__"]:
    print("Interrupt:", pending.id, pending.value)
    print("Response schema:", pending.response_schema)

In [ ]:
# Execute
per_call_result = per_call_paused
while per_call_result.get("__interrupt__"):
    pending = per_call_result["__interrupt__"][0]
    per_call_choice = input(f"{pending.value['args']['label']} by ID (approve / reject): ").strip().lower()
    if per_call_choice not in {"approve", "reject"}:
        raise ValueError("Choose approve or reject; this action remains paused.")
    per_call_result = per_call_agent.invoke(Command(resume={pending.id: {"type": per_call_choice}}), config=per_call_config)

In [ ]:
# Inspect output
show_messages(per_call_result["messages"], pretty=True)
print("Executed labels:", reviewed_labels)